In [1]:
import subprocess
import sys
import pyspark
print(f"PySpark Version: {pyspark.__version__}")

PySpark Version: 4.0.2


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NF-ToN-IoT-v2 Attack Classification") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print("SparkSession created successfully!")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")
spark

SparkSession created successfully!
Spark UI available at: http://dd5772b7ed46:4040


In [5]:
DATA_PATH = "/content/NF-ToN-IoT-v2-train.csv"

start = time.time()
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("mode", "DROPMALFORMED") \
    .csv(DATA_PATH)

load_time = time.time() - start
print(f"Dataset loaded in {load_time:.2f} seconds")
print(f"Total Rows: {df_raw.count():,}")
print(f"Total Columns: {len(df_raw.columns)}")

Dataset loaded in 23.85 seconds
Total Rows: 1,322,443
Total Columns: 45


In [6]:
df_raw.printSchema()

root
 |-- IPV4_SRC_ADDR: string (nullable = true)
 |-- L4_SRC_PORT: integer (nullable = true)
 |-- IPV4_DST_ADDR: string (nullable = true)
 |-- L4_DST_PORT: integer (nullable = true)
 |-- PROTOCOL: integer (nullable = true)
 |-- L7_PROTO: double (nullable = true)
 |-- IN_BYTES: integer (nullable = true)
 |-- IN_PKTS: integer (nullable = true)
 |-- OUT_BYTES: integer (nullable = true)
 |-- OUT_PKTS: integer (nullable = true)
 |-- TCP_FLAGS: integer (nullable = true)
 |-- CLIENT_TCP_FLAGS: integer (nullable = true)
 |-- SERVER_TCP_FLAGS: integer (nullable = true)
 |-- FLOW_DURATION_MILLISECONDS: integer (nullable = true)
 |-- DURATION_IN: integer (nullable = true)
 |-- DURATION_OUT: integer (nullable = true)
 |-- MIN_TTL: integer (nullable = true)
 |-- MAX_TTL: integer (nullable = true)
 |-- LONGEST_FLOW_PKT: integer (nullable = true)
 |-- SHORTEST_FLOW_PKT: integer (nullable = true)
 |-- MIN_IP_PKT_LEN: integer (nullable = true)
 |-- MAX_IP_PKT_LEN: integer (nullable = true)
 |-- SRC_TO

In [8]:
from pyspark.sql.types import StringType, DoubleType, FloatType, IntegerType, LongType

numeric_cols = [f.name for f in df_raw.schema.fields
                if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))]

string_cols = [f.name for f in df_raw.schema.fields
               if isinstance(f.dataType, StringType)]

print(f"Numeric columns : {len(numeric_cols)}")
print(f"String columns  : {len(string_cols)}")
print(f"\nNumeric : {numeric_cols}")
print(f"\nString  : {string_cols}")

Numeric columns : 42
String columns  : 3

Numeric : ['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Label']

String  : ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Attack']


In [10]:
from pyspark.sql.functions import isnan, when, col, sum as spark_sum, count

total_rows = df_raw.count()

numeric_nulls = [
    spark_sum(
        when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)
    ).alias(c)
    for c in numeric_cols
]

string_nulls = [
    spark_sum(
        when(col(c).isNull() | (col(c) == ""), 1).otherwise(0)
    ).alias(c)
    for c in string_cols
]

print("NULL AND MISSING VALUES - NUMERIC COLUMNS")
null_num_df = df_raw.select(numeric_nulls).toPandas().T.reset_index()
null_num_df.columns = ["Column", "Null_Count"]
null_num_df["Null_Percent"] = (null_num_df["Null_Count"] / total_rows * 100).round(2)
print(null_num_df.to_string(index=False))

print("\nNULL AND MISSING VALUES - STRING COLUMNS")
null_str_df = df_raw.select(string_nulls).toPandas().T.reset_index()
null_str_df.columns = ["Column", "Null_Count"]
null_str_df["Null_Percent"] = (null_str_df["Null_Count"] / total_rows * 100).round(2)
print(null_str_df.to_string(index=False))

NULL AND MISSING VALUES - NUMERIC COLUMNS
                     Column  Null_Count  Null_Percent
                L4_SRC_PORT           0           0.0
                L4_DST_PORT           0           0.0
                   PROTOCOL           0           0.0
                   L7_PROTO           0           0.0
                   IN_BYTES           0           0.0
                    IN_PKTS           0           0.0
                  OUT_BYTES           0           0.0
                   OUT_PKTS           0           0.0
                  TCP_FLAGS           0           0.0
           CLIENT_TCP_FLAGS           0           0.0
           SERVER_TCP_FLAGS           0           0.0
 FLOW_DURATION_MILLISECONDS           0           0.0
                DURATION_IN           0           0.0
               DURATION_OUT           0           0.0
                    MIN_TTL           0           0.0
                    MAX_TTL           0           0.0
           LONGEST_FLOW_PKT           0 

In [12]:
import pandas as pd

print("BASIC STATISTICS - NUMERIC COLUMNS")
stats_df = df_raw.select(numeric_cols).describe().toPandas().set_index("summary")
chunk_size = 10
col_chunks = [numeric_cols[i:i+chunk_size] for i in range(0, len(numeric_cols), chunk_size)]
for i, chunk in enumerate(col_chunks):
    print(f"\nColumns {i*chunk_size+1} to {i*chunk_size+len(chunk)}")
    print(stats_df[chunk].T.to_string())

print("\nSTRING COLUMN UNIQUE VALUES")
string_summary = []
for c in string_cols:
    unique_vals = df_raw.select(c).distinct().toPandas()[c].tolist()
    string_summary.append({
        "Column": c,
        "Unique_Count": len(unique_vals),
        "Sample_Values": str(unique_vals[:5])
    })
print(pd.DataFrame(string_summary).to_string(index=False))

BASIC STATISTICS - NUMERIC COLUMNS

Columns 1 to 10
summary             count                mean              stddev  min       max
L4_SRC_PORT       1322443   43375.41144457644  15149.912911726047    0     65535
L4_DST_PORT       1322443   9270.299865476243  16840.332057184005    0     65535
PROTOCOL          1322443   7.617758950669329   3.926732820184019    1        58
L7_PROTO          1322443  13.758874947352158  31.998528080323528  0.0     248.0
IN_BYTES          1322443   708.8335497257726   64478.44066156779    4  46530772
IN_PKTS           1322443   6.067170380878419  351.83871152111743    1    236880
OUT_BYTES         1322443   757.8352586841172   40815.25725375912    0  22779856
OUT_PKTS          1322443   3.072989913364886  162.49331938972733    0    122471
TCP_FLAGS         1322443   12.79887753196168  11.295044445260691    0       223
CLIENT_TCP_FLAGS  1322443    9.49496953743942  10.925330077399146    0       223

Columns 11 to 20
summary                       count    

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import os

drive_path = "/content/drive/MyDrive/BigDataProject/NF-ToN-IoT-v2-train.csv"
print(f"File exists : {os.path.exists(drive_path)}")
print(f"File size   : {os.path.getsize(drive_path) / (1024**3):.2f} GB")

File exists : True
File size   : 1.98 GB


In [20]:
import time

DATA_PATH    = "/content/drive/MyDrive/BigDataProject/NF-ToN-IoT-v2-train.csv"
PARQUET_PATH = "/content/drive/MyDrive/BigDataProject/parquet/ton_iot_data"

print("Loading full dataset...")
start = time.time()

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("mode", "DROPMALFORMED") \
    .csv(DATA_PATH)

print(f"Rows loaded : {df_raw.count():,}")
print(f"Load time   : {time.time() - start:.2f} seconds")

print("Saving to Parquet...")
start = time.time()

df_raw.write \
    .mode("overwrite") \
    .partitionBy("Label") \
    .parquet(PARQUET_PATH)

print(f"Save time       : {time.time() - start:.2f} seconds")

df_check = spark.read.parquet(PARQUET_PATH)
print(f"Parquet rows    : {df_check.count():,}")
print(f"Parquet columns : {len(df_check.columns)}")
print("Parquet saved successfully")

Loading full dataset...
Rows loaded : 13,552,396
Load time   : 173.08 seconds
Saving to Parquet...
Save time       : 252.02 seconds
Parquet rows    : 13,552,396
Parquet columns : 45
Parquet saved successfully
